# Q2 Appendix: Random-vs-Time Split — All Four Metrics (raw means over runs)

This appendix reproduces the article-ready Q2 tables (random vs. time-split inflation),
reporting all four metrics as raw mean-over-runs values instead of only
Macro-F1 (`f1_score`), in **tidy/long format**.

**Table format:**
- Metrics are reported in exactly four columns, in order: **Accuracy, Precision,
  Recall, Macro-F1** (mean over runs).
- The split is a single **`Split`** label column on the left, with values
  `Random split` and `Time split` (fixed order: Random first, then Time). Each source
  group therefore yields two rows.
- No between-condition difference or ranking columns (the former
  `Δ MacroF1 (Rand − Time)` reference is dropped).

**Q2 invariants (identical to the main analysis):** the split
*"Random with same distribution"* is dropped; **in-network only** (train set = test set,
renamed `network`); all 11 models; averaged over `run_no`. The load/melt/average setup
carries **all four metrics** through the `metric` column.

Tables (CSV + LaTeX) are written to `tables/` with the prefix `q2_appendix_`.


Setup: imports, visual theme, paths, model list, and metric ordering.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths (notebook runs from A/)
DATA_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR   = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

# Style (repo convention)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

NETWORKS = ['SetA', 'SetB', 'SetC', 'SetD']
# Metric order for expansion: Accuracy, Precision, Recall, Macro-F1
METRIC_ORDER = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}
PRIMARY = 'f1_score'

ALL_MODELS = ['gru', 'knn', 'lightgbm', 'logistic-regression', 'lstm',
              'mlp', 'nn', 'random-forest', 'rnn', 'svm', 'xgboost']


Load the results CSV and reshape wide → long, keeping **in-network** rows only
(train set = test set). This matches `Q2/Q2_analysis.ipynb` verbatim, except that
**all four metrics are carried through** the `metric` column (not filtered to
`f1_score`).

In [2]:
# ── Load raw data ──────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH, index_col=0)
raw = raw[raw['split'] != "Random with same distribution"]

# ── Parse metric columns into long form ───────────────────────────────────────
metric_cols = [c for c in raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'run_no']

long = raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='metric_key', value_name='value'
)
long[['train_set', 'test_set', 'metric']] = long['metric_key'].str.split('/', expand=True)
long = long.drop(columns='metric_key')

# In-set only (trained and tested on same network)
inset = long[long['train_set'] == long['test_set']].copy()
inset = inset.rename(columns={'train_set': 'network'}).drop(columns='test_set')

print(f"Long-form rows (all):    {len(long):,}")
print(f"Long-form rows (in-set): {len(inset):,}")
print(inset['split'].value_counts())


Long-form rows (all):    38,528
Long-form rows (in-set): 9,632
split
Time split      4864
Random split    4768
Name: count, dtype: int64


Average across runs (`run_no`) for each unique (model, task, split, approach,
network, **metric**) configuration. All four metrics are retained.

In [3]:
# ── Average across runs ────────────────────────────────────────────────────────
group_keys = ['model', 'task', 'split', 'enable_sequences', 'network', 'metric']

avg = (inset.groupby(group_keys, as_index=False)['value']
            .agg(mean_value='mean'))

print("Averaged rows:", len(avg))
print(avg.head(3))


Averaged rows: 1088
  model    task         split  enable_sequences network     metric  mean_value
0   gru  Binary  Random split              True    SetA   accuracy    0.941879
1   gru  Binary  Random split              True    SetA   f1_score    0.801669
2   gru  Binary  Random split              True    SetA  precision    0.801336


### Expansion helper

`expand_metrics` takes the run-averaged data plus a set of grouping keys, and returns a
**tidy/long** table: one row per (grouping keys × split), with the split carried as a
single `Split` label column and the four metrics (Accuracy / Precision / Recall /
Macro-F1) as mean-over-runs columns.

Metric column order: Accuracy, Precision, Recall, Macro-F1. Split order: `Random split`
first, then `Time split`.

In [4]:
SPLIT_ORDER = ['Random split', 'Time split']

def expand_metrics(data, index_keys):
    """Pivot metric into columns, keeping split as a single 'Split' label column (tidy/long).

    Returns a DataFrame with columns:
      <index_keys...>, Split, Accuracy, Precision, Recall, Macro-F1
    with one row per (index_keys x split); splits ordered Random then Time.
    """
    # Mean over the (already run-averaged) values within each index x split x metric group
    g = (data.groupby(index_keys + ['split', 'metric'], as_index=False)['mean_value']
             .mean())
    wide = g.pivot_table(index=index_keys + ['split'], columns='metric',
                         values='mean_value')

    # Order the four metric columns and relabel
    wide = wide.reindex(columns=METRIC_ORDER)
    wide.columns = [METRIC_LABEL[m] for m in METRIC_ORDER]

    out = wide.reset_index()
    out = out.rename(columns={'split': 'Split'})
    # Fixed split order: Random first, then Time
    out['Split'] = pd.Categorical(out['Split'], categories=SPLIT_ORDER, ordered=True)
    sort_cols = index_keys + ['Split']
    out = out.sort_values(sort_cols).reset_index(drop=True)
    out['Split'] = out['Split'].astype(str)
    return out

def save_table(df, name, caption):
    """Round to 4 dp, save CSV + LaTeX, display inline."""
    out = df.round(4)
    out.to_csv(TAB_DIR / f'q2_appendix_{name}.csv', index=False)
    latex = out.to_latex(index=False, float_format='%.4f', escape=False,
                         caption=caption, label=f'tab:q2_appendix_{name}')
    (TAB_DIR / f'q2_appendix_{name}.tex').write_text(latex)
    print(f'Saved q2_appendix_{name} -> CSV + TEX   shape={out.shape}')
    return out


### q2_appendix_overall

Source: `table1_overall_gap`. Grouped by `task`, including the **All** summary rows
(mean over both tasks). One row per (task × split); columns `task, Split, Accuracy,
Precision, Recall, Macro-F1`.

In [5]:
# ── q2_appendix_overall (source: table1_overall_gap) ─────────────────────────
per_task = expand_metrics(avg, ['task'])

# "All" summary rows: mean over both tasks (equivalently, over all in-network configs)
all_rows = expand_metrics(avg.assign(task='All'), ['task'])
t_overall = pd.concat([per_task, all_rows], ignore_index=True)

out_overall = save_table(t_overall, 'overall',
    "Means over runs of Accuracy/Precision/Recall/Macro-F1 (in-network), random split vs.\ time split. "
    "Overall (averaged over all models and networks); the \'\'All\'\' rows average over both tasks.")
display(out_overall)


Saved q2_appendix_overall -> CSV + TEX   shape=(6, 6)


,task,Split,Accuracy,Precision,Recall,Macro-F1
0,Binary,Random split,0.9381,0.8833,0.8576,0.8606
1,Binary,Time split,0.9271,0.8464,0.7692,0.7834
2,Multiclass,Random split,0.9413,0.8619,0.7500,0.7794
3,Multiclass,Time split,0.9186,0.7728,0.6652,0.6753
4,All,Random split,0.9397,0.8726,0.8038,0.8200
5,All,Time split,0.9229,0.8096,0.7172,0.7293


### q2_appendix_per_network

Source: `table2_per_network_gap`. Grouped by (`network`, `task`). One row per
(network × task × split); columns `network, task, Split, Accuracy, Precision, Recall,
Macro-F1`.

In [6]:
# ── q2_appendix_per_network (source: table2_per_network_gap) ─────────────────
t_per_network = expand_metrics(avg, ['network', 'task'])

out_per_network = save_table(t_per_network, 'per_network',
    "Means over runs of Accuracy/Precision/Recall/Macro-F1 (in-network), random split vs.\ time split. "
    "Per network and task.")
display(out_per_network)


Saved q2_appendix_per_network -> CSV + TEX   shape=(16, 7)


,network,task,Split,Accuracy,Precision,Recall,Macro-F1
0,SetA,Binary,Random split,0.9112,0.8080,0.7675,0.7684
1,SetA,Binary,Time split,0.8959,0.6591,0.5656,0.5664
2,SetA,Multiclass,Random split,0.9212,0.7643,0.6368,0.6670
3,SetA,Multiclass,Time split,0.8911,0.5424,0.4834,0.4578
4,SetB,Binary,Random split,0.9335,0.8771,0.8496,0.8518
5,SetB,Binary,Time split,0.9297,0.8845,0.7866,0.8021
6,SetB,Multiclass,Random split,0.9444,0.8978,0.7774,0.8099
7,SetB,Multiclass,Time split,0.9276,0.8625,0.7344,0.7615
8,SetC,Binary,Random split,0.9261,0.8760,0.8558,0.8585
9,SetC,Binary,Time split,0.8971,0.8595,0.7717,0.8003


### q2_appendix_per_model

Source: `table3_per_model_gap`. Grouped by `model`, one row per (model × split).
Columns `model, Split, Accuracy, Precision, Recall, Macro-F1`. Models are ordered by
overall mean Macro-F1 (across both splits) **descending**, then by Split (Random, Time).

In [7]:
# ── q2_appendix_per_model (source: table3_per_model_gap) ─────────────────────
t_per_model = expand_metrics(avg, ['model'])

# Sort models by overall mean Macro-F1 (across both splits) descending, then Split order
model_rank = (t_per_model.groupby('model')['Macro-F1'].mean()
              .sort_values(ascending=False))
model_order = list(model_rank.index)
t_per_model['model'] = pd.Categorical(t_per_model['model'],
                                      categories=model_order, ordered=True)
t_per_model['Split'] = pd.Categorical(t_per_model['Split'],
                                      categories=SPLIT_ORDER, ordered=True)
t_per_model = t_per_model.sort_values(['model', 'Split']).reset_index(drop=True)
t_per_model['model'] = t_per_model['model'].astype(str)
t_per_model['Split'] = t_per_model['Split'].astype(str)

out_per_model = save_table(t_per_model, 'per_model',
    "Means over runs of Accuracy/Precision/Recall/Macro-F1 (in-network), random split vs.\ time split. "
    "Per model (averaged over networks and tasks), sorted by overall mean Macro-F1 descending.")
display(out_per_model)


Saved q2_appendix_per_model -> CSV + TEX   shape=(22, 6)


,model,Split,Accuracy,Precision,Recall,Macro-F1
0,gru,Random split,0.9802,0.9164,0.8657,0.8843
1,gru,Time split,0.9717,0.8542,0.7748,0.7948
2,rnn,Random split,0.9791,0.9290,0.8450,0.8751
3,rnn,Time split,0.9713,0.8598,0.7696,0.7902
4,lstm,Random split,0.9789,0.9290,0.8422,0.8732
5,lstm,Time split,0.9789,0.8661,0.7588,0.7850
6,lightgbm,Random split,0.9395,0.9045,0.8616,0.8757
7,lightgbm,Time split,0.9042,0.8012,0.7435,0.7453
8,xgboost,Random split,0.9385,0.9025,0.8593,0.8730
9,xgboost,Time split,0.9069,0.8241,0.7352,0.7446
